In [ ]:
"""
Notebook 01 v3 — Build the GEE assets for the analysis pipeline

Purpose:
    Rebuild every GEE asset that downstream notebooks depend on, updated for
    the v3 methodology.

What changes from v2:
    * Baseline period reduced to 2016-2017. Analysis of the KKL fire scar
      dataset showed that 2018 experienced 1,178 recorded fires (an order of
      magnitude more than any other year), reflecting the peak of the
      incendiary balloon campaign. Including it in the baseline would
      contaminate the reference distribution.
    * Sentinel-2 collection extended to start from 2016 (was 2017) so the
      new baseline has real observations to average.
    * The fire-reference inventory is now the union of KKL polygons (long time series)
      and the Layer 2 October 2023 polygons (Planet + Sentinel-2 mapping).
    * The infrastructure mask keeps only human_infrastructure_mask; the
      FCODE filter on otef_landuse is dropped, because the LC layer sampled
      in Notebook 02 already carries the land-cover information the model
      needs.

Outputs (all under projects/YOUR-GEE-PROJECT/assets/):
    * fires_layer1_layer2_10m                — merged fire FeatureCollection
    * s2_monthly_image_collection_v3_10m     — 120 monthly S2 composites
    * fire_monthly_labels_v3_10m             — 108 monthly binary labels
    * seasonal_baseline_mean_2016_2017_10m   — 12 per-calendar-month means
    * seasonal_baseline_median_2016_2017_10m — 12 per-calendar-month medians
    * seasonal_baseline_std_2016_2017_10m    — 12 per-calendar-month stddevs
"""

# Notebook 01 v3 — Build GEE assets

Rebuilds Sentinel-2 composites, monthly fire labels, and seasonal
baselines for the v3 pipeline. Baseline period: **2016-2017**. Fire
labels: **Layer 1 + Layer 2 merged**.

In [ ]:
VERSION = "_10m"

PROJECT_ID = "YOUR-GEE-PROJECT"
ASSET_ROOT = f"projects/{PROJECT_ID}/assets"

# Spatial parameters
SCALE      = 10
CRS        = "EPSG:32636"
MAX_PIXELS = 1e13

# Cloud masking and temporal coverage
CLEAR_THRESH = 0.60
DATE_START   = "2016-01-01"
DATE_END     = "2025-12-31"

# Baseline period — see the discussion above on why 2018 is excluded
BASELINE_START = "2016-01-01"
BASELINE_END   = "2017-12-31"
BASELINE_YEARS = [2016, 2017]

# Output asset paths — v3 suffix keeps the pre-v3 assets available for
# side-by-side comparison and rollback
ASSET_COLLECTION       = f"{ASSET_ROOT}/s2_monthly_image_collection_v3{VERSION}"
LABEL_COLLECTION       = f"{ASSET_ROOT}/fire_monthly_labels_v3{VERSION}"
BASELINE_MEAN_FOLDER   = f"{ASSET_ROOT}/seasonal_baseline_mean_2016_2017{VERSION}"
BASELINE_MEDIAN_FOLDER = f"{ASSET_ROOT}/seasonal_baseline_median_2016_2017{VERSION}"
BASELINE_STD_FOLDER    = f"{ASSET_ROOT}/seasonal_baseline_std_2016_2017{VERSION}"

# Source assets
ROI_FC_ID           = f"{ASSET_ROOT}/gaza_envelope_for_decell_Dissolve"
HUMAN_INFRA_FC_ID   = f"{ASSET_ROOT}/human_infrastructure_mask_utm36n"
FIRES_LAYER1_FC_ID  = f"{ASSET_ROOT}/fires_scars_otef_6991"
FIRES_LAYER2_FC_ID  = f"{ASSET_ROOT}/fire_damage"

# Merged fire asset built by this notebook
FIRES_MERGED_FC_ID  = f"{ASSET_ROOT}/fires_layer1_layer2{VERSION}"

print("=" * 60)
print(f"VERSION           = '{VERSION}'")
print(f"Resolution        = {SCALE} m")
print(f"S2 date range     = {DATE_START} → {DATE_END}")
print(f"Baseline years    = {BASELINE_YEARS}")
print("=" * 60)
print(f"S2 composites     -> {ASSET_COLLECTION}")
print(f"Fire labels       -> {LABEL_COLLECTION}")
print(f"Baseline mean     -> {BASELINE_MEAN_FOLDER}")
print(f"Baseline median   -> {BASELINE_MEDIAN_FOLDER}")
print(f"Baseline std      -> {BASELINE_STD_FOLDER}")
print(f"Merged fires      -> {FIRES_MERGED_FC_ID}")
print("=" * 60)

In [ ]:
# Install and initialise Earth Engine
!pip install -q -U geemap

import ee
import geemap

ee.Authenticate()
ee.Initialize(project=PROJECT_ID)

print("Earth Engine initialised for project:", PROJECT_ID)

## Optional cleanup

Uncomment and run once to delete the previous v2 baseline collections
before rebuilding. The v2 S2 collection and fire labels can be kept in
place (they use different asset names).

In [ ]:
# ============================================================
# Uncomment to delete v2 baselines before rebuilding
# ============================================================
'''
paths_to_delete = [
    f"{ASSET_ROOT}/seasonal_baseline_mean{VERSION}",
    f"{ASSET_ROOT}/seasonal_baseline_median{VERSION}",
    f"{ASSET_ROOT}/seasonal_baseline_std{VERSION}",
]

for path in paths_to_delete:
    print(f"Processing: {path}")
    try:
        result = ee.data.listAssets({"parent": path})
        images = result.get("assets", [])
        print(f"  Found {len(images)} images")
        for img in images:
            ee.data.deleteAsset(img["name"])
        ee.data.deleteAsset(path)
        print(f"  Deleted: {path}\n")
    except Exception as e:
        print(f"  Error: {e}\n")

print("Cleanup complete.")
'''

## Pre-flight check

In [ ]:
# Report whether each target v3 asset already exists.
paths_to_check = [
    ASSET_COLLECTION,
    LABEL_COLLECTION,
    BASELINE_MEAN_FOLDER,
    BASELINE_MEDIAN_FOLDER,
    BASELINE_STD_FOLDER,
    FIRES_MERGED_FC_ID,
]

print(f"Checking v3 target paths:\n")
collisions = []
for path in paths_to_check:
    try:
        info = ee.data.getAsset(path)
        try:
            n_imgs = ee.ImageCollection(path).size().getInfo()
        except Exception:
            n_imgs = "n/a"
        print(f"  EXISTS  {path}  (type={info['type']}, n={n_imgs})")
        collisions.append(path)
    except Exception:
        print(f"  FREE    {path}")

print()
if collisions:
    print(f"{len(collisions)} v3 asset(s) already exist — delete them "
          f"before re-running the corresponding step below.")
else:
    print(f"All v3 target paths are free.")

## Step 1 — Load ROI, masks, and both fire sources

The ROI and the human infrastructure mask are unchanged from v2. The
FCODE filter on otef_landuse is dropped: the land-cover layer sampled
in Notebook 02 provides the same discriminative signal, and keeping
built-up pixels in-scope matches the research decision to allow the
model to see the full landscape context.

In [ ]:
working_area = ee.FeatureCollection(ROI_FC_ID)
region       = working_area.geometry()

human_infra_fc = ee.FeatureCollection(HUMAN_INFRA_FC_ID)

# Human-infrastructure mask: 1 where infrastructure is present, 0 elsewhere.
# We take the complement (Not) below to KEEP non-infrastructure pixels.
human_infra_mask = (human_infra_fc
                    .reduceToImage(["Shape_Area"], ee.Reducer.count())
                    .gt(0))

# Final spatial mask: exclude only human infrastructure. Land-cover based
# exclusion (roads, buildings that WorldCover missed) is handled downstream
# via the LC layer sampled in Notebook 02.
final_mask = human_infra_mask.Not()

# Load both fire layers
fires_l1 = ee.FeatureCollection(FIRES_LAYER1_FC_ID)
fires_l2 = ee.FeatureCollection(FIRES_LAYER2_FC_ID)

print(f"Fire Layer 1 (KKL, fires_scars_otef_6991):     "
      f"{fires_l1.size().getInfo()} polygons")
print(f"Fire Layer 2 (Planet+S2, fire_damage):         "
      f"{fires_l2.size().getInfo()} polygons")

## Step 2 — Merge the two fire layers into a single FeatureCollection

Both layers carry fire dates as Unix millisecond timestamps but under
different property names:

* Layer 1 uses `FireDate`
* Layer 2 uses `FirstDateW` (the earliest date the burn scar is visible)

We harmonise both to a single `fire_date` property (also Unix ms), then
concatenate. Layer 2 features get a marker (`source = 'layer2'`) so
provenance is preserved in the merged asset.

In [ ]:
def prepare_layer1(f):
    """Attach normalised properties to a Layer 1 fire polygon."""
    fire_ms = ee.Number(f.get("FireDate"))
    date    = ee.Date(fire_ms)
    return f.set({
        "fire_date":         fire_ms,
        "fire_year":         date.get("year"),
        "fire_month":        date.get("month"),
        "source":            "layer1",
        "system:time_start": fire_ms,
    })


def prepare_layer2(f):
    """Attach normalised properties to a Layer 2 fire polygon."""
    fire_ms = ee.Number(f.get("FirstDateW"))
    date    = ee.Date(fire_ms)
    return f.set({
        "fire_date":         fire_ms,
        "fire_year":         date.get("year"),
        "fire_month":        date.get("month"),
        "source":            "layer2",
        "system:time_start": fire_ms,
    })


# Restrict to valid geometries and to the analysis date range.
DATE_START_MS = ee.Date(DATE_START).millis()
DATE_END_MS   = ee.Date(DATE_END).millis()

l1_norm = (fires_l1
           .filterBounds(region)
           .map(lambda f: f.set("geom_type", f.geometry().type()))
           .filter(ee.Filter.inList("geom_type", ["Polygon", "MultiPolygon"]))
           .map(prepare_layer1)
           .filter(ee.Filter.rangeContains("fire_date",
                                           DATE_START_MS, DATE_END_MS)))

l2_norm = (fires_l2
           .filterBounds(region)
           .map(lambda f: f.set("geom_type", f.geometry().type()))
           .filter(ee.Filter.inList("geom_type", ["Polygon", "MultiPolygon"]))
           .map(prepare_layer2)
           .filter(ee.Filter.rangeContains("fire_date",
                                           DATE_START_MS, DATE_END_MS)))

# Merge into one collection. Overlapping polygons across sources are kept
# separately — the downstream monthly-label rasteriser paints them together.
fires_merged = l1_norm.merge(l2_norm)

print(f"Layer 1 within date range: {l1_norm.size().getInfo()}")
print(f"Layer 2 within date range: {l2_norm.size().getInfo()}")
print(f"Merged fires:              {fires_merged.size().getInfo()}")

## Step 3 — Export the merged fire FeatureCollection to GEE

Persisting the merged collection as an asset means downstream notebooks
do not need to re-run the merge every time, and the merge logic is
frozen at export time.

In [ ]:
task_merge = ee.batch.Export.table.toAsset(
    collection  = fires_merged,
    description = "fires_layer1_layer2_export",
    assetId     = FIRES_MERGED_FC_ID,
)
task_merge.start()

print(f"Export task started: fires_layer1_layer2_export")
print(f"Target: {FIRES_MERGED_FC_ID}")
print(f"Wait ~5 minutes for completion, then continue to Step 4.")
print(f"Check status: https://code.earthengine.google.com/tasks")

## Step 4 — Build the clean Sentinel-2 collection

Bands include the 10 m visible/NIR bands (B4, B8) and the 20 m red-edge
and SWIR bands (B8A, B11, B12). Cloud Score+ (cs_cdf) is used to mask
cloudy pixels at a threshold of CLEAR_THRESH. The infrastructure mask
from Step 1 is applied to every image.

**Difference from v2:** the collection now starts in 2016. The 2016
images are used only by the baseline in Step 6; the model in later
notebooks reads months from 2017 onwards.

In [ ]:
BANDS = ["B4", "B8", "B8A", "B11", "B12"]

s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
      .filterBounds(region)
      .filterDate(DATE_START, DATE_END)
      .select(BANDS))

csplus = (ee.ImageCollection("GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED")
          .filterBounds(region)
          .filterDate(DATE_START, DATE_END))

s2_with_cs = s2.linkCollection(csplus, ["cs_cdf"])


def mask_with_cs(img):
    return (img
            .updateMask(img.select("cs_cdf").gte(CLEAR_THRESH))
            .select(BANDS)
            .updateMask(final_mask)
            .toFloat())


s2_clean = ee.ImageCollection(s2_with_cs.map(mask_with_cs))

print(f"Clean Sentinel-2 images over ROI: {s2_clean.size().getInfo()}")

# Verify the native projection of a SWIR band
img_first = s2_clean.first()
proj      = img_first.select("B11").projection()
print(f"CRS:       {proj.crs().getInfo()}")
print(f"Transform: {proj.transform().getInfo()}")

## Step 5 — Build and export monthly composites

For each (year, month) in 2016-2025, compute a mean composite of the
clean Sentinel-2 images that fell in that month.

**Storage optimisation:** only the 5 raw bands (B4, B8, B8A, B11, B12)
plus `pct_valid` are exported, cast to FLOAT (float32). The 7 spectral
indices (NDVI, EVI, NDMI, NBR, MIRBI, BSI, BAIS2) are analytical
functions of the raw bands and are recomputed on the fly in Notebook 02
during sampling. Storing them in every monthly asset roughly doubles the
GEE storage footprint without adding any information; casting the raw
bands to FLOAT (instead of the default DOUBLE) halves the raw-band size
too. Together these reduce each monthly asset from ~2.1 GB to ~0.7 GB.

In [ ]:
def monthly_mean_image(year, month):
    """Build a single monthly mean composite with only raw bands + pct_valid.

    The output has 6 bands: B4, B8, B8A, B11, B12 (all FLOAT reflectance in
    [0, 1]) plus pct_valid. Indices and anomalies are computed on the fly
    in Notebook 02.
    """
    start = ee.Date.fromYMD(year, month, 1)
    end   = start.advance(1, "month")

    ic = s2_clean.filterDate(start, end)

    # Rescale to surface reflectance in [0, 1] and cast to FLOAT
    ic_ref = ic.map(lambda img:
                    img.select(BANDS).multiply(0.0001).toFloat()
                       .copyProperties(img, img.propertyNames()))

    # Monthly mean composite of the raw bands only
    comp = ic_ref.select(BANDS).mean().toFloat()

    # Fraction of valid (cloud-free) observations in the month
    # Guard against empty months with .max(1) to prevent divide-by-zero
    n_obs_valid       = ic_ref.select("B4").count()
    n_images_in_month = ee.Number(ic_ref.size()).max(1)
    pct_valid         = (n_obs_valid.divide(n_images_in_month)
                         .rename("pct_valid").toFloat())

    return (comp
            .addBands(pct_valid)
            .set({
                "year":              year,
                "month":             month,
                "month_start":       start.format("YYYY-MM-dd"),
                "n_images_in_month": ic_ref.size(),
                "system:time_start": start.millis(),
            })
            .clip(region))


def export_month_to_asset(year, month):
    img      = monthly_mean_image(year, month)
    ym       = f"{int(year)}_{int(month):02d}"
    desc     = f"s2_monthly_mean_{ym}"
    asset_id = f"{ASSET_COLLECTION}/{desc}"

    task = ee.batch.Export.image.toAsset(
        image            = img,
        description      = desc,
        assetId          = asset_id,
        region           = region,
        scale            = SCALE,
        crs              = CRS,
        maxPixels        = MAX_PIXELS,
        pyramidingPolicy = {"default": "mean"},
    )
    task.start()
    return task


# Submit exports for all months in 2016-2025 = 120 tasks
tasks_s2 = []
for y in range(2016, 2026):
    for m in range(1, 13):
        tasks_s2.append(export_month_to_asset(y, m))

n_bands = len(BANDS) + len(INDICES) + 1
print(f"Submitted {len(tasks_s2)} monthly composite tasks")
print(f"Bands per image: {n_bands} ({len(BANDS)} raw + {len(INDICES)} "
      f"indices + 1 quality)")
print(f"Target: {ASSET_COLLECTION}")
print(f"Wait for these tasks to finish before continuing to Step 6.")

## Step 6 — Compute per-calendar-month baseline statistics

From the composites exported in Step 5, restrict to the baseline period
(2016 and 2017) and compute per-band mean, median, and standard deviation
for each calendar month across the two reference years. This produces three
ImageCollections of twelve images each.

**Run this cell only after all Step 5 export tasks have completed.**

In [ ]:
# Indices needed for downstream anomaly computation. These are not stored
# in the S2 collection — they are computed on the fly here from the raw
# bands before the reference-period aggregation, so that the baseline
# statistics (mean, median, std) are per-index and can drive anomaly
# calculation in Notebook 02.
INDICES = ["NDVI", "EVI", "NDMI", "NBR", "MIRBI", "BSI", "BAIS2"]
BASELINE_BANDS = BANDS + INDICES  # 5 raw + 7 indices = 12 bands


def add_indices(img_ref):
    """Attach seven spectral indices to a raw Sentinel-2 composite."""
    eps = ee.Image.constant(1e-6)

    b4  = img_ref.select("B4")
    b8  = img_ref.select("B8")
    b8a = img_ref.select("B8A")
    b11 = img_ref.select("B11")
    b12 = img_ref.select("B12")

    ndvi = img_ref.normalizedDifference(["B8", "B4"]).rename("NDVI")
    ndmi = img_ref.normalizedDifference(["B8", "B11"]).rename("NDMI")
    nbr  = img_ref.normalizedDifference(["B8", "B12"]).rename("NBR")

    # Two-band EVI variant — drops the Blue term, appropriate when the
    # surface reflectance product is atmospherically corrected
    evi = (b8.subtract(b4).multiply(2.5)
             .divide(b8.add(b4.multiply(2.4)).add(1).add(eps))
             .rename("EVI"))

    # Mid-Infrared Burn Index (Trigg & Flasse 2001)
    mirbi = (b12.multiply(10).subtract(b11.multiply(9.8)).add(2)).rename("MIRBI")

    # Bare Soil Index — B8A substituted for the Blue band
    bsi = ((b11.add(b4).subtract(b8.add(b8a)))
           .divide(b11.add(b4).add(b8).add(b8a).add(eps))
           .rename("BSI"))

    # Sentinel-2 Burned Area Index (Filipponi 2018)
    term1 = ee.Image(1).subtract(
        b11.multiply(b12).multiply(b8a).divide(b4.add(eps)).sqrt())
    term2 = b12.subtract(b8a).divide(b12.add(b8a).add(eps).sqrt()).add(1)
    bais2 = term1.multiply(term2).rename("BAIS2")

    return (img_ref.select(BANDS)
                    .addBands([ndvi, evi, ndmi, nbr, mirbi, bsi, bais2])
                    .toFloat())


ic_full = ee.ImageCollection(ASSET_COLLECTION)

# Reference period for baseline computation — 2016 and 2017 only
# Each image in ic_ref has only raw bands + pct_valid; we compute
# indices on the fly with add_indices before aggregating.
ic_ref = (ic_full
          .filter(ee.Filter.inList("year", BASELINE_YEARS))
          .map(add_indices))
print("Images in reference period 2016-2017:", ic_ref.size().getInfo())

baseline_tasks = []

for month in range(1, 13):
    ic_month = ic_ref.filter(ee.Filter.eq("month", month))
    n = ic_month.size().getInfo()
    print(f"  Month {month:02d}: {n} images")

    baseline_mean = (ic_month.select(BASELINE_BANDS).mean().toFloat()
                     .rename(BASELINE_BANDS)
                     .set({
                         "month":             month,
                         "baseline_type":     "mean",
                         "reference_years":   "2016-2017",
                         "system:time_start": ee.Date.fromYMD(
                             2017, month, 1).millis(),
                     })
                     .clip(region))

    baseline_median = (ic_month.select(BASELINE_BANDS).median().toFloat()
                       .rename(BASELINE_BANDS)
                       .set({
                           "month":             month,
                           "baseline_type":     "median",
                           "reference_years":   "2016-2017",
                           "system:time_start": ee.Date.fromYMD(
                               2017, month, 1).millis(),
                       })
                       .clip(region))

    baseline_std = (ic_month.select(BASELINE_BANDS)
                    .reduce(ee.Reducer.stdDev()).toFloat()
                    .rename([b + "_stdDev" for b in BASELINE_BANDS])
                    .rename(BASELINE_BANDS)
                    .set({
                        "month":             month,
                        "baseline_type":     "std",
                        "reference_years":   "2016-2017",
                        "system:time_start": ee.Date.fromYMD(
                            2017, month, 1).millis(),
                    })
                    .clip(region))

    mm = f"{month:02d}"

    t1 = ee.batch.Export.image.toAsset(
        image       = baseline_mean,
        description = f"baseline_mean_{mm}",
        assetId     = f"{BASELINE_MEAN_FOLDER}/baseline_mean_{mm}",
        region      = region, scale = SCALE, crs = CRS, maxPixels = MAX_PIXELS,
    )
    t1.start()

    t2 = ee.batch.Export.image.toAsset(
        image       = baseline_median,
        description = f"baseline_median_{mm}",
        assetId     = f"{BASELINE_MEDIAN_FOLDER}/baseline_median_{mm}",
        region      = region, scale = SCALE, crs = CRS, maxPixels = MAX_PIXELS,
    )
    t2.start()

    t3 = ee.batch.Export.image.toAsset(
        image       = baseline_std,
        description = f"baseline_std_{mm}",
        assetId     = f"{BASELINE_STD_FOLDER}/baseline_std_{mm}",
        region      = region, scale = SCALE, crs = CRS, maxPixels = MAX_PIXELS,
    )
    t3.start()

    baseline_tasks.extend([t1, t2, t3])

print(f"\nSubmitted {len(baseline_tasks)} baseline export tasks")
print(f"  Mean:   {BASELINE_MEAN_FOLDER}")
print(f"  Median: {BASELINE_MEDIAN_FOLDER}")
print(f"  Std:    {BASELINE_STD_FOLDER}")

## Step 7 — Build and export monthly binary fire labels

For every month in the analysis period (2017-01 through 2025-12), paint
the merged fire polygons active in that month onto a binary image at
10 m resolution. Pixel value 1 means at least one fire polygon (from
either Layer 1 or Layer 2) overlapped the pixel during that month.

**Run this cell only after Step 3 (fire merge) has completed.**

In [ ]:
# Reload the persisted merged fire asset so this step is idempotent
# against re-runs; matches how downstream notebooks will read it.
fires_merged_asset = ee.FeatureCollection(FIRES_MERGED_FC_ID)


def monthly_fire_label_image(year, month):
    """Rasterise fires active in the given calendar month."""
    start = ee.Date.fromYMD(year, month, 1)
    end   = start.advance(1, "month")

    fires_m = fires_merged_asset.filter(
        ee.Filter.rangeContains("fire_date", start.millis(), end.millis()))

    return (ee.Image(0).toByte()
            .paint(fires_m, 1)
            .rename("burned_this_month")
            .clip(region)
            .unmask(0)
            .set({
                "year":              year,
                "month":             month,
                "month_start":       start.format("YYYY-MM-dd"),
                "n_fire_polygons":   fires_m.size(),
                "system:time_start": start.millis(),
            }))


def export_monthly_fire_label_to_asset(year, month):
    img      = monthly_fire_label_image(year, month)
    ym       = f"{int(year)}_{int(month):02d}"
    desc     = f"fire_label_{ym}"
    asset_id = f"{LABEL_COLLECTION}/{desc}"

    task = ee.batch.Export.image.toAsset(
        image       = img,
        description = desc,
        assetId     = asset_id,
        region      = region,
        scale       = SCALE,
        crs         = CRS,
        maxPixels   = MAX_PIXELS,
    )
    task.start()
    return task


# 108 monthly labels covering the analysis period 2017-01 → 2025-12
tasks_labels = []
for y in range(2017, 2026):
    for m in range(1, 13):
        tasks_labels.append(export_monthly_fire_label_to_asset(y, m))

print(f"Submitted {len(tasks_labels)} monthly fire label tasks")
print(f"Target: {LABEL_COLLECTION}")

## Done

After all export tasks in Steps 3, 5, 6, and 7 have completed, the
following v3 assets will exist:

* `fires_layer1_layer2_10m` — merged fire polygons
* `s2_monthly_image_collection_v3_10m` — 120 monthly composites
* `fire_monthly_labels_v3_10m` — 108 monthly binary labels
* `seasonal_baseline_{mean,median,std}_2016_2017_10m` — 12 images each

Continue to Notebook 02 to sample pixels and extract per-pixel time
series with land-cover features attached.